# Door: train and evaluate ExtraTrees (300 trees, minimum leaf 3)

Run the cells in order, or select **Run All**. This notebook loads the raw Train dataset, builds features, trains new models, and prints the measured scores. All implementation is in this notebook. No saved model or result file is required.

**Local data:** copy `Door` from the supplied `PS3/02_Datasets` bundle into this folder's `data/`, retaining the Train names below. Data stays local and is ignored by Git.

```text
Door/
  train.ipynb
  data/
    Train.csv
    Train_Segments_Answer.csv
```

Use Python 3.11. Install the pinned CPU libraries once in the notebook's Python environment:
```python
%pip install numpy==1.26.4 pandas==2.2.1 scipy==1.12.0 scikit-learn==1.4.1.post1 openpyxl==3.1.5 rainflow==3.2.0 threadpoolctl==3.4.0
```

The displayed scores are cross-validation on labelled **Train** data. The recipe was selected in earlier experiments on this corpus, so these are exploratory validation results. Official Test is never read. The supplied stream has long gaps between every labelled cycle, making segmentation unusually easy; the score does not demonstrate performance during continuous idle operation.

## 1. Imports and data location

In [1]:
from pathlib import Path
import sys, time, hashlib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from threadpoolctl import threadpool_limits
HERE = Path.cwd() if Path.cwd().name == 'Door' else Path.cwd()/'Door'
DATA = HERE/'data'
assert DATA.is_dir(), f'Place the Door Train dataset in {DATA} first (see the cell above).'
SEED = 17
started = time.perf_counter()
print('Python:', sys.version.split()[0], '| NumPy:', np.__version__, '| pandas:', pd.__version__, '| sklearn:', sklearn.__version__)
print('Data:', DATA.resolve())
from datetime import datetime, timezone
from sklearn.ensemble import ExtraTreesClassifier

Python: 3.11.4 | NumPy: 1.26.4 | pandas: 2.2.1 | sklearn: 1.4.1.post1
Data: C:\000NebulaX\nebulax_p3\Door\data


## 2. Load the stream and labelled events
Labels define evaluation blocks and training targets, while the detector below finds cycle boundaries using only raw timestamps.

In [2]:
def timestamp(value):
    year,month,day,hour,minute,second,millis=map(int,value.split('-'))
    return datetime(year,month,day,hour,minute,second,millis*1000,tzinfo=timezone.utc).timestamp()

frame=pd.read_csv(DATA/'Train.csv')
labels=pd.read_csv(DATA/'Train_Segments_Answer.csv')
times=np.array([timestamp(t) for t in frame.Datetime])
raw=frame.iloc[:,1:].to_numpy(float)
assert raw.shape[1]==16 and np.isfinite(raw).all() and np.all(np.diff(times)>0)
gold=[]
coverage=np.zeros(len(frame),int)
for r in labels.itertuples(index=False):
    a=int(np.searchsorted(times,timestamp(r.start_time)))
    b=int(np.searchsorted(times,timestamp(r.end_time),side='right')-1)
    assert times[a]==timestamp(r.start_time) and times[b]==timestamp(r.end_time) and b-a+1==r.n_rows
    coverage[a:b+1]+=1
    gold.append({'a':a,'b':b,'start':r.start_time,'end':r.end_time,'label':r.status})
assert np.all(coverage==1)
blocks=[list(block) for block in np.array_split(np.arange(len(gold)),4)]
print(f'{len(frame):,} readings; {len(gold)} labelled cycles; four chronological folds')
display(labels.status.value_counts().rename('cycles').to_frame())

18,036 readings; 110 labelled cycles; four chronological folds


,cycles
status,
Normal,80
Abnormal resistance,30


## 3. Detect cycles, extract features, and define the score
Use the retained 0.1-second gap rule. Each event supplies 13 statistics for each of 16 telemetry channels, plus duration and sample count. No event ID, absolute timestamp, or label is a feature.

In [3]:
def detect(a,b):
    cuts=np.flatnonzero(np.diff(times[a:b+1])>0.1)+1
    endpoints=np.r_[0,cuts,b-a+1]
    return [{'a':a+int(x),'b':a+int(y)-1,'start':str(frame.iloc[a+int(x)].Datetime),
             'end':str(frame.iloc[a+int(y)-1].Datetime)} for x,y in zip(endpoints[:-1],endpoints[1:])]

def events_in(block_ids):
    events,truth=[],[]
    for block in block_ids:
        ids=blocks[block]
        events.extend(detect(gold[ids[0]]['a'],gold[ids[-1]]['b']))
        truth.extend(gold[i] for i in ids)
    return events,truth

def features(events):
    rows=[]
    for e in events:
        v=raw[e['a']:e['b']+1]
        stats=[v.mean(0),v.std(0),v.min(0),v.max(0),np.quantile(v,.25,axis=0),np.median(v,axis=0),
               np.quantile(v,.75,axis=0),np.sqrt((v**2).mean(0)),np.abs(v).mean(0),v[0],v[-1],
               v[-1]-v[0],np.abs(np.diff(v,axis=0)).sum(0)]
        rows.append(np.r_[np.concatenate(stats),times[e['b']]-times[e['a']],len(v)])
    return np.asarray(rows)

def iou(a,b):
    a0,a1,b0,b1=timestamp(a['start']),timestamp(a['end']),timestamp(b['start']),timestamp(b['end'])
    intersection=max(0,min(a1,b1)-max(a0,b0))
    union=a1-a0+b1-b0-intersection
    return intersection/union if union>0 else 0

def targets(events,truth):
    output=[]
    for event in events:
        overlaps=[iou(event,t) for t in truth]
        j=int(np.argmax(overlaps))
        assert overlaps[j]>.99, 'A fitting event needs an unambiguous label'
        output.append(int(truth[j]['label']=='Abnormal resistance'))
    return np.array(output)

def score(truth,predictions):
    pairs=[(iou(t,p),i,j) for i,t in enumerate(truth) for j,p in enumerate(predictions)
           if t['label']==p['label'] and iou(t,p)>0]
    used_true,used_pred=set(),set()
    credit=0.
    for overlap,i,j in sorted(pairs,key=lambda row:(-row[0],row[1],row[2])):
        if i not in used_true and j not in used_pred:
            used_true.add(i);used_pred.add(j);credit+=overlap
    return 2*credit/(len(truth)+len(predictions)) if truth or predictions else 0.

def predict_events(model,events):
    return [dict(e,label='Abnormal resistance' if p else 'Normal') for e,p in zip(events,model.predict(features(events)))]

def estimator():
    return ExtraTreesClassifier(n_estimators=300,min_samples_leaf=3,class_weight='balanced',
                                max_features='sqrt',n_jobs=2,random_state=SEED)
print('Feature extractor and IoU-weighted F1 ready.')

Feature extractor and IoU-weighted F1 ready.


## 4. Train four folds and evaluate the held-out raw regions

In [4]:
models,fold_rows,oof=[],[],[]
for fold in range(4):
    fit_events,fit_truth=events_in([k for k in range(4) if k!=fold])
    val_events,val_truth=events_in([fold])
    assert not ({e['a'] for e in fit_events}&{e['a'] for e in val_events})
    model=estimator().fit(features(fit_events),targets(fit_events,fit_truth))
    fit_pred=predict_events(model,fit_events)
    val_pred=predict_events(model,val_events)
    majority=int(np.mean(targets(fit_events,fit_truth))>=.5)
    baseline=[dict(e,label='Abnormal resistance' if majority else 'Normal') for e in val_events]
    row={'fold':fold+1,'fit_score':score(fit_truth,fit_pred),'validation_score':score(val_truth,val_pred),
         'majority_score':score(val_truth,baseline),'fit_events':len(fit_truth),'validation_events':len(val_truth)}
    fold_rows.append(row);models.append(model);oof.extend(val_pred)
    print(f"Fold {fold+1}: train={row['fit_score']:.6f}, validation={row['validation_score']:.6f}")
display(pd.DataFrame(fold_rows))

Fold 1: train=1.000000, validation=1.000000


Fold 2: train=1.000000, validation=1.000000


Fold 3: train=1.000000, validation=1.000000


Fold 4: train=1.000000, validation=1.000000


,fold,fit_score,validation_score,majority_score,fit_events,validation_events
0,1,1.0,1.0,0.714286,82,28
1,2,1.0,1.0,0.821429,82,28
2,3,1.0,1.0,0.666667,83,27
3,4,1.0,1.0,0.703704,83,27


## 5. Printed results and trained model
`models` contains the four newly trained fold models. For later inference, average their abnormal-resistance probabilities on features produced by the same detector.

In [5]:
validation_score=score(gold,oof)
train_score=np.average([r['fit_score'] for r in fold_rows],weights=[r['fit_events'] for r in fold_rows])
print(f'Train-fit IoU-weighted F1: {train_score:.9f}')
print(f'Out-of-fold IoU-weighted F1: {validation_score:.9f}')
print(f'Trained models: {len(models)} | Held-out events: {len(oof)}')
print(f'Elapsed: {time.perf_counter()-started:.1f} seconds')
display(pd.DataFrame(oof)[['start','end','label']].head(10))

Train-fit IoU-weighted F1: 1.000000000
Out-of-fold IoU-weighted F1: 1.000000000
Trained models: 4 | Held-out events: 110
Elapsed: 3.8 seconds


,start,end,label
0,2023-7-5-0-0-0-0,2023-7-5-0-0-3-700,Normal
1,2023-7-5-0-0-23-999,2023-7-5-0-0-26-839,Normal
2,2023-7-5-0-0-51-266,2023-7-5-0-0-53-986,Abnormal resistance
3,2023-7-5-0-1-33-989,2023-7-5-0-1-37-709,Abnormal resistance
4,2023-7-5-0-2-24-608,2023-7-5-0-2-28-308,Abnormal resistance
5,2023-7-5-0-2-59-77,2023-7-5-0-3-1-937,Normal
6,2023-7-5-0-3-56-723,2023-7-5-0-3-59-503,Normal
7,2023-7-5-0-4-37-800,2023-7-5-0-4-40-680,Normal
8,2023-7-5-0-5-4-993,2023-7-5-0-5-8-713,Abnormal resistance
9,2023-7-5-0-5-45-986,2023-7-5-0-5-48-826,Normal
